### Positional Encodings

In [1]:
import math 
import random

In [ ]:
### Absolute positional encodings
def sinusoidal_pe(n, d, base = 10000.0):
    pe = [[0.0] * d for _ in range(n)]
    for pos in range(n):
        for i in range(d // 2):
            theta = pos / (base ** (2 * i /d))
            pe[pos][2 * i] = math.sin(theta)
            pe[pos][2 * i + 1] = math.cos(theta)
    
    return pe

In [4]:
def apply_rope(x, pos, base = 10000.0):
    d = len(x)
    out = list(x)
    for i in range(d // 2):
        theta = pos / (base ** (2 * i / d))
        c = math.cos(theta)
        s = math.sin(theta)
        a = x[2 * i]
        b = x[2 * i + 1]
        out[2 * i] = a * c - b * s
        out[2 * i + 1] = a * s + b * c
    return out

In [8]:
def demo_sinusoidal():
    print("=== sinusoidal positional encoding ===")
    pe = sinusoidal_pe(n=8, d=8)
    print("first 4 positions, first 4 dims:")
    for pos in range(4):
        print(f"  pos={pos}: " + "  ".join(f"{v:+.3f}" for v in pe[pos][:4]))
    print()

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def demo_rope_relative():
    print("=== RoPE: dot product depends only on relative distance ===")
    rng = random.Random(0)
    d = 16
    q = [rng.gauss(0, 1) for _ in range(d)]
    k = [rng.gauss(0, 1) for _ in range(d)]

    pairs = [(3, 5), (7, 9), (100, 102), (1024, 1026)]
    print(f"{'pos_q':>6}  {'pos_k':>6}  {'gap':>4}  {'<q_rot, k_rot>':>18}")
    for pq, pk in pairs:
        q_rot = apply_rope(q, pq)
        k_rot = apply_rope(k, pk)
        d_prod = dot(q_rot, k_rot)
        print(f"{pq:>6}  {pk:>6}  {pk - pq:>4}  {d_prod:>18.6f}")
    print("all rows with gap=2 should have matching dot products.")
    print()


def demo_rope_base_scaling():
    print("=== RoPE base scaling (NTK-aware for long context) ===")
    rng = random.Random(1)
    d = 8
    q = [rng.gauss(0, 1) for _ in range(d)]
    k = [rng.gauss(0, 1) for _ in range(d)]

    for base in [10000, 100000, 1_000_000]:
        q_rot = apply_rope(q, pos=4096, base=base)
        k_rot = apply_rope(k, pos=4098, base=base)
        print(f"  base={base:>8d}  score={dot(q_rot, k_rot):+.6f}")
    print("larger base = slower rotation = longer context without phase wrap.")
    print()



In [9]:
demo_sinusoidal()

=== sinusoidal positional encoding ===
first 4 positions, first 4 dims:
  pos=0: +0.000  +1.000  +0.000  +1.000
  pos=1: +0.841  +0.540  +0.100  +0.995
  pos=2: +0.909  -0.416  +0.199  +0.980
  pos=3: +0.141  -0.990  +0.296  +0.955



In [10]:
demo_rope_relative()

=== RoPE: dot product depends only on relative distance ===
 pos_q   pos_k   gap      <q_rot, k_rot>
     3       5     2           -6.530165
     7       9     2           -6.530165
   100     102     2           -6.530165
  1024    1026     2           -6.530165
all rows with gap=2 should have matching dot products.



In [11]:
demo_rope_base_scaling()

=== RoPE base scaling (NTK-aware for long context) ===
  base=   10000  score=+1.331200
  base=  100000  score=+1.368584
  base= 1000000  score=+1.388781
larger base = slower rotation = longer context without phase wrap.

